### Benchmarking FFT vs. two different wavelet transform implementaitons 

#### Run SEVERAL TIMES to get a decent comparision!

In [4]:
import numpy as np
import pywt
import time
import matplotlib.pyplot as plt


In [5]:

# 1. Generate a non-stationary signal (sine wave that jumps in frequency)
fs = 1000  # Sampling rate
t = np.linspace(0, 1, fs, endpoint=False)
# 50Hz for the first half, 150Hz for the second half
sig = np.concatenate([np.sin(2 * np.pi * 50 * t[:500]), 
                      np.sin(2 * np.pi * 150 * t[500:])])

# --- Fast Fourier Transform (FFT) ---
# Best for global frequency data; loses exact "when" the jump happened.
start = time.time()
fft_res = np.fft.fft(sig)
fft_time = time.time() - start

# --- Fast Wavelet Transform (FWT/DWT) ######## IT IS FASTER THEN fCWT below!!!!! ##### ---
# Discrete approach; O(N) complexity. Breaks signal into approximation/detail.
start = time.time()
# 'db4' is a common Daubechies wavelet
coeffs = pywt.wavedec(sig, 'db4', level=4) 
fwt_time = time.time() - start

# --- Fast Continuous Wavelet Transform (fCWT) --####### IT IS ACTUALLY NOT THAT FAST COMPARED TO THE ABOVE !!!! ##### ---
# Provides high-resolution time-frequency maps. 
# Libraries like 'fcwt' or specialized PyWavelets routines use FFT-based 
# optimizations to make this nearly as fast as DWT.
widths = np.arange(1, 31)
start = time.time()
cwtmatr, freqs = pywt.cwt(sig, widths, 'mexh') # Mexican Hat wavelet
fcwt_time = time.time() - start

print(f"FFT Speed:  {fft_time:.6f}s")
print(f"FWT Speed:  {fwt_time:.6f}s")
print(f"fCWT Speed: {fcwt_time:.6f}s")

FFT Speed:  0.000276s
FWT Speed:  0.000265s
fCWT Speed: 0.001680s


### I got a decent GPU and  Biiiiig data
### Can I run it on GPU?

In [7]:
# If you have CUDA GPU use this

# import torch
# import ptwt  # pip install ptwt

# # 1. Generate signal (1 million points)
# N = 2**20
# data_np = np.random.randn(N).astype(np.float32)
# data_torch = torch.from_numpy(data_np).cuda()  # Move to GPU

# wavelet = 'db4'
# level = 4

# # --- CPU Implementation (PyWavelets) ---
# start = time.time()
# coeffs_cpu = pywt.wavedec(data_np, wavelet, mode='zero', level=level)
# cpu_time = time.time() - start

# # --- GPU Implementation (ptwt) ---
# # Note: ptwt accepts the same wavelet string names as pywt
# start = time.time()
# coeffs_gpu = ptwt.wavedec(data_torch, wavelet, mode='zero', level=level)
# torch.cuda.synchronize()  # Wait for GPU to finish
# gpu_time = time.time() - start

# print(f"PyWavelets (CPU) Time: {cpu_time:.6f}s")
# print(f"ptwt (GPU) Time:       {gpu_time:.6f}s")

# # 2. Verification: Convert GPU result back to verify it matches CPU
# approx_gpu = coeffs_gpu[0].cpu().numpy()
# print(f"Results Match: {np.allclose(coeffs_cpu[0], approx_gpu, atol=1e-5)}")


#### Run SEVERAL TIMES to get a decent comparision!

In [14]:
# Use this if you have Mac with M-type processor (Apple Silicon)

# 1. Setup Signal (1 million points)
N = 2**20
data_np = np.random.randn(N).astype(np.float32)

# Check for Apple Silicon GPU (MPS)
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
data_torch = torch.from_numpy(data_np).to(device)

wavelet = 'db4'
level = 4

# --- CPU Implementation (PyWavelets) ---
# Uses standard CPU threads
start = time.time()
coeffs_cpu = pywt.wavedec(data_np, wavelet, mode='zero', level=level)
cpu_time = time.time() - start

# --- Mac GPU Implementation (ptwt + MPS) ---
# Uses the M4's GPU cores via Metal
start = time.time()
coeffs_gpu = ptwt.wavedec(data_torch, wavelet, mode='zero', level=level)

# Important: Synchronize to get accurate timing on Mac
torch.mps.synchronize() 
mps_time = time.time() - start

print(f"Device used:          {device}")
print(f"PyWavelets (CPU) Time: {cpu_time:.6f}s")
print(f"ptwt (MPS GPU) Time:   {mps_time:.6f}s")

# 2. Verification
# Flattening ptwt output to match pywt's list-based coefficients
flat_gpu = torch.cat([c.cpu() if isinstance(c, torch.Tensor) else torch.tensor(c) for c in coeffs_gpu])
print(f"Results Match: {np.allclose(coeffs_cpu[0], coeffs_gpu[0].cpu().numpy(), atol=1e-5)}")



Device used:          mps
PyWavelets (CPU) Time: 0.004352s
ptwt (MPS GPU) Time:   0.003617s
Results Match: True
